# Validation v2 source-backed curation

This workbench builds 144 deterministic candidates from pinned MIT/Apache-2.0 sources and checkpoints human decisions in Google Drive. Calibration/confirmatory assignments are intentionally hidden. Approval permits edits but requires a substantive source-grounded rationale.

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !git clone -q https://github.com/ashioyajotham/safety_governor.git /content/safety_governor
    %cd /content/safety_governor
    !pip -q install -r requirements-review.txt
else:
    REPO_ROOT = Path.cwd().resolve()


## Build the immutable candidate set

The fetcher verifies exact upstream SHA-256 values. Re-running the builder with seed 42 produces the same candidate IDs, source groups, and hidden role assignments.

In [ ]:
!python -m scripts.fetch_corpus_sources
CANDIDATES = Path('/content/validation_v2_candidates.jsonl') if IN_COLAB else Path('data/working/validation_v2/candidates.jsonl')
if not CANDIDATES.exists():
    !python -m scripts.build_validation_v2_candidates --output {CANDIDATES}
print(CANDIDATES, sum(1 for line in CANDIDATES.open() if line.strip()))


## Human review

Inspect source fidelity, edit both responses where necessary, and approve only a natural contrast that isolates the declared archetype. The destination is durable; a runtime disconnect does not erase saved rows.

In [ ]:
REVIEWER = 'REPLACE_WITH_REVIEWER_NAME'
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DECISIONS = Path('/content/drive/MyDrive/safety_governor/validation_v2_curation_decisions.jsonl')
else:
    DECISIONS = Path('data/working/validation_v2/curation_decisions.jsonl')
from safety_governor.validation_v2_widgets import build_validation_v2_curation_widget
panel = build_validation_v2_curation_widget(CANDIDATES, DECISIONS, reviewer=REVIEWER)


## Freeze after all 144 decisions

Download the decision file and run from a clean repository root:

```bash
python -m scripts.materialize_validation_v2 data/working/validation_v2/candidates.jsonl PATH/TO/validation_v2_curation_decisions.jsonl
```

The command fails before writing if quotas, provenance, source isolation, lexical-diversity, or frozen-corpus overlap checks fail.